In [ ]:
# !pip install pylabwons
!pip install pylabwons --upgrade --no-deps

In [ ]:
from datetime import datetime
import pandas as pd
import pandas_datareader.data as web
import plotly.io as pio
import plotly.graph_objects as go
import pylabwons as lw
import yfinance as yf
pio.renderers.default = "vscode"

clock = datetime.now()

# ASSET




| 구분 | **기초 자산** | **가중 방식** | **운용 방식** | **주요 종목** | **주요 특징** |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `XLE` (대형주 중심) | S&P 500 에너지 섹터 기업 | **시가총액 가중** (대형주 비중⬆️) | 1배수 (현물 ETF) | 엑손모빌, 쉐브론 등 대기업 | 안정적인 대형주 및 배당 중심 |
| `XOP` (탐사/생산 중심) | S&P Oil & Gas E&P 지수 | **동일 가중** (모든 종목 균등 비중) | 1배수 (현물 ETF) | 중소형 탐사·생산(E&P) 기업 | 유가 변동에 민감한 중소형주 포함 |
| `ERX` (🧲 XLE) | Energy Select Sector 지수 | **시가총액 가중** (대형주 비중⬆️) | **일일 수익률 정방향 2배 (ETF)** | 엑손모빌, 쉐브론 등 대기업 | 대형 에너지주 상승 시 레버리지 수익 |
| `ERY` (🧲 XLE) | Energy Select Sector 지수 | **시가총액 가중** (대형주 비중⬆️) | **일일 수익률 역방향 -2배 (ETF)** | 엑손모빌, 쉐브론 등 대기업 | 대형 에너지주 하락 시 인버스 수익 |
| `GUSH` (🧲 XOP) | S&P Oil & Gas E&P 지수 | **동일 가중** (모든 종목 균등 비중) | **일일 수익률 정방향 2배 (ETF)** | 중소형 탐사·생산(E&P) 기업 | 유가 상승기 중소형주 강세 시 높은 수익 |
| `DRIP` (🧲 XOP) | S&P Oil & Gas E&P 지수 | **동일 가중** (모든 종목 균등 비중) | **일일 수익률 역방향 -2배 (ETF)** | 중소형 탐사·생산(E&P) 기업 | 유가 하락기 중소형주 약세 시 높은 수익 |
| `WTIU` | Solactive MicroSectors US Big Oil Index | **동일 가중** (상위 12개 대형주 중심) | **일일 수익률 3배 추종 (ETN)** | 엑손모빌, 쉐브론 등 우량주 12개 | 단기 방향성 매매용 (변동성↗️) |
| `USO` (원유 선물) | WTI(서부 텍사스산 원유) 선물 | **해당 없음** (선물 계약 중심) | 1배수 (선물 ETF) | WTI 원유 선물 | 원유 가격(선물) 직접 추종 (롤오버 비용 발생) |



In [ ]:
TICKERS = lw.DataDict(**{
    ticker: yf.Ticker(ticker) for ticker in [
        'XLE', 
        'XOP', 
        'ERX', 
        'ERY', 
        'GUSH', 
        'DRIP', 
        'WTIU', 
        'USO'
]})

objs = {}
for ticker in TICKERS.values():
    ohlcv = ticker.history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
    ohlcv.columns = ['open', 'high', 'low', 'close', 'volume']
    objs[ticker.ticker] = ohlcv
    

asset = pd.concat(objs, axis=1)
asset = asset.tz_localize(None) if asset.index.tz is not None else asset
# asset

In [ ]:
mar = lw.MultiAssetRelation(asset.xs('close', axis=1, level=1))
# mar.cummulative_return
# mar.drawdown
fig = mar.plotly_multi('cumulative_return')
fig.update_layout(height=600)

# MACRO

## INDICATORS LIST

| 번호 | 지표명 (Indicator) | 데이터 출처 (Source) | 업데이트 주기 | 분석 내재 의미 및 활용법 (Core Interpretation) |
| :--- | :--- | :--- | :--- | :--- |
| **1** | **상업용 원유 재고**<br>(Commercial Crude Inventories) | 미국 에너지정보청<br>([EIA](https://eia.gov)) | 매주 수요일<br>(주간) | - 미국 내 단기 원유 공급 과잉 및 부족을 가늠하는 척도<br>- 계절성(5년 평균치) 대비 급감 시 유가 상방 압력 |
| **2** | **SPR 원유 재고**<br>(Strategic Petroleum Reserve) | 미국 에너지정보청<br>([EIA](https://eia.gov)) | 매주 수요일<br>(주간) | - 정부의 인위적 시장 개입(방출/재비축)을 나타내는 지표<br>- 상업용 재고와 합산한 '총 재고량'으로 공급 실태 파악 필수 |
| **3** | **Baker Hughes Rig Count**<br>(시추기 수) | 베이커 휴즈 공시<br>([Baker Hughes](https://bakerhughes.com)) | 매주 금요일<br>(주간) | - 미국 셰일오일 공급의 최선행 지표 (유가 변동에 3~6개월 후행)<br>- 유가 상승에도 Rig Count 증가가 둔화되면 공급 부족 지속 시그널 |
| **4** | **`CL=F`, `BZ=F`, `NG=F`**<br>(WTI, Brent, 천연가스 선물) | 야후 파이낸스<br>([Yahoo Finance](https://yahoo.com)) | 실시간<br>(일간/분간) | - 에너지 원자재 시장의 벤치마크 가격 지표<br>- 근월물과 원월물 간의 가격 차이(백워데이션/콘탱고)로 시장 심리 진단 |
| **5** | **생산자 물가지수: Drilling PPI**<br>(PCU213111213111) | 세인트루이스 연은<br>([FRED](https://stlouisfed.org)) | 매월 중순<br>(월간) | - 유전 시추 및 개발에 드는 물리적 비용 지수<br>- 유가가 높아도 이 지표가 급등하면 기업의 실질 마진(FCF) 훼손 의미 |
| **6** | **`DXY` 달러 인덱스**<br>(U.S. Dollar Index) | 야후 파이낸스<br>([Yahoo Finance](https://yahoo.comquote/DX-Y.NYB/)) | 실시간<br>(일간/분간) | - 원유의 결제 통화인 달러화의 절대 가치<br>- 유가와 강한 역상관관계 (`DXY` 하락 = 유가 상승 지지 환경) |
| **7** | **북미 정제시설 가동률**<br>(Refinery Utilization Rate) | 미국 에너지정보청<br>([EIA](https://eia.gov)) | 매주 수요일<br>(주간) | - 정유사들이 원유를 사들이는 '실제 수요'의 강도를 증명<br>- 90% 이상 유지 시 원유 소화력 양호, 하락 시 원유 수요 둔화 경고 |
| **8** | **Crack Spread (3-2-1)**<br>(정제마진 직접 계산) | Yahoo Finance 데이터 가공<br>(`CL=F`, `RB=F`, `HO=F` 활용) | 실시간<br>(일간) | - 정유사(XLE 등 주요 기업)의 핵심 수익성 직결 지표<br>- 유가 고점 대비 마진 선행 하락 시 에너지 주식 탈출(Exit) 신호 |

---

## CHECK-LIST & GUIDELINES

본 가이드라인은 대형주 중심(XLE) 및 생산/탐사 중심(XOP) ETF 투자 시, 레버리지/인버스 진입 및 청산 타이밍을 계량화하기 위한 매크로 지표 조합 분석 템플릿입니다.

---

<h3>LONG & LEVERAGE</h3>

아래 지표 조합 중 **4개 이상이 충족**되거나, **핵심 조합(수요-정제-가격)이 일치**할 경우 1배수 진입 또는 단기 레버리지(2x, 3x) 전략 검토


| 대분류 | ☑️ 체크 항목 | 시장 판단 및 투자 시그널 |
| :--- | :--- | :--- |
| ‼️**단기 수요 및 정제 마진** | ⬜ Crack Spread 상승 + 정제시설 가동률(Utilization Rate) 상승 | - 최종 제품(휘발유, 등유 등) 수요 강력<br>- 정제 마진 개선 국면<br>- **XLE, XOP 강력한 매수 신호** |
| ‼️**단기 수요 및 정제 마진** | ⬜ 상업용 원유 재고 감소 (EIA) | - 5년 평균 대비 재고 하락 추세 또는 예상치보다 큰 감소 폭<br>- **단기 유가(CL=F, BZ=F) 상승 압력 작용** |
| **중장기 공급 및 생산 기조** | ⬜ Baker Hughes Rig Count 정체/감소 + 생산자 물가지수(PPI: Drilling) 상승 | - 시추 비용(PPI) 상승에도 시추기 수(Rig Count) 정체<br>- 공급 과잉 우려 해소 및 고유가 유지 환경 조성 (**XOP 호재**) |
| **중장기 공급 및 생산 기조** | ⬜ SPR(전략비축유) 재고 고정 또는 매입 전환 (EIA) | - 미 정부의 SPR 방출 종료 및 재충전(매입) 단계 진입<br>- **유가 하방 지지선 형성** |
| **매크로 및 금융 자산 동향** | ⬜ 달러 인덱스(DXY) 하락 추세 전환 | - 원유의 달러 결제 특성 반영<br>- **CL=F, BZ=F 가격의 명목적 상승 유발** |
| **매크로 및 금융 자산 동향** | ⬜ 천연가스 선물(NG=F) 계절적 반등 또는 CL=F 동반 상승 | - **에너지 섹터 전반의 투자 심리(Sentiment) 개선** |
| **레버리지 진입** | ⬜ 상업용 원유 재고 감소 + SPR 재고 증가 동시 발생 (EIA) | - 정부(SPR) 매입과 민간 재고(상업용) 고갈 동시 발생<br>- 최상의 공급 부족 신호로 **가격 상승 지속성 담보** |
| **레버리지 진입** | ⬜ BZ=F(브렌트)와 CL=F(WTI) 스프레드(Spread) 확대 | - 글로벌 기준물(브렌트)이 미국 유가보다 강세<br>- 미국 외 글로벌 실물 수요 탄탄 (수출 마진 확대로 **미국 에너지 기업 호재**) |
| **레버리지 진입** | ⬜ Crack Spread 상승 속 원유 선물(CL=F, BZ=F) 백워데이션(Backwardation) 심화 | - 근월물 가격이 원월물보다 비싼 현상 심화 (실물 원유 극심한 부족)<br>- **숏 스퀴즈 유발 가능성 상승으로 레버리지 진입 최적기** |
| **레버리지 진입** | ⬜ DXY 하락 속 NG=F 지지선 확인 및 Drilling PPI 최고치 경신 | - 인플레이션(PPI)으로 인한 에너지 개발 단가의 구조적 상승<br>- 달러 약세가 겹치며 **자산 가격 하방 완벽 지지** |




🟩 레버리지 진입 필터


- ☑️ **DXY 하락 속 NG=F(천연가스)의 지지선 확인 및 Drilling PPI 최고치 경신**
  * **판단:** 인플레이션 압력(PPI)으로 인해 에너지 개발 단가 자체가 구조적으로 올라간 상태에서 달러까지 약세이므로, 자산 가격의 하방이 완벽하게 지지됩니다.

---

<h3>SHORT / INVERSE / CASH OUT: EXIT</h3>

아래 위험 신호 중 **3개 이상이 동시 발생**하거나, **정제 마진 붕괴가 확인**되면 포지션을 축소(수익 실현)하거나 단기 인버스 상품 진입을 검토합니다.


| 대분류 | ☑️ 체크 항목 | 시장 판단 및 매도/인버스 시그널 |
| :--- | :--- | :--- |
| ‼️**정제 및 수요 둔화 우려** | ⬜ Crack Spread 급락 (정제시설 가동률 고점 통과) | - 유가가 높더라도 정제 마진 붕괴 시 정유사(XLE) 실적 악화<br>- **원유 수요 감소의 전조 증상** |
| ‼️**정제 및 수요 둔화 우려** | ⬜ 상업용 원유 재고(EIA)의 지속적인 누적 | - 공급이 수요를 초과하기 시작했다는 확실한 증거<br>- **지표 발표 시마다 유가 하방 압력 강화** |
| **공급 과잉 및 생산 단가 하락** | ⬜ Baker Hughes Rig Count 급증 + 생산자 물가지수(PPI: Drilling) 안정화 | - 시추 비용 하락 및 공급(Rig) 증가 국면<br>- **셰일 기업 중심인 XOP의 급락 유발 가능** |
| **공급 과잉 및 생산 단가 하락** | ⬜ SPR 원유 재고의 대규모 방출 지속 | - 정부의 강제적인 시장 공급 물량 확대 구간<br>- **유가 상방 유의미하게 제한** |
| **역풍(Headwind) 및 자산 붕괴** | ⬜ 달러 인덱스(DXY) 강한 반등 또는 고점 돌파 | - 글로벌 유동성 위축 발생<br>- **원유 가격의 명목적 하락 압력으로 작용** |
| **역풍(Headwind) 및 자산 붕괴**| ⬜ CL=F, BZ=F 선물 가격의 데드크로스 (기술적 추세 이탈) | - 매크로 지표 악화와 함께 주요 이평선(50일, 200일) 이탈<br>- **레버리지 포지션 즉시 청산 신호** |
| **인버스 진입** | ⬜ 정제시설 가동률 하락 속 상업용 원유 재고 증가 | - 가동률을 낮춤에도 재고가 쌓이는 원유 수요 둔화 상태<br>- **유가 폭락 신호탄으로 즉시 현금화 또는 인버스 준비** |
| **인버스 진입** | ⬜ CL=F, BZ=F 가격 상승 속 Crack Spread 나홀로 급락 (다이버전스) | - 제품 소비자가 고유가를 버티지 못해 마진이 깨지는 현상<br>- 가짜 상승일 확률이 높으며 **정유주(XLE) 급락 임박 시사** |
| **인버스 진입** | ⬜ Baker Hughes Rig Count 증가 전환 + 원유 선물 콘탱고(Contango) 진입 | - 시추기 수 증가 및 콘탱고(원월물>근월물) 변동 발생<br>- 공급 과잉 우려가 시장을 지배하는 **확실한 숏(Short) 신호** |
| **인버스 진입** | ⬜ DXY 급등과 동시에 Drilling PPI 하락 전환 | - 마진 한계선인 생산 비용 하락 및 달러 강세 동시 진행<br>- 섹터 마진 스프레드가 압박받아 **주가 폭락 유발** |



🟥 정제 및 수요 둔화 우려 (최우선 탈출 신호)
- ☑️ **Crack Spread 급락 (정제시설 가동률 고점 통과)**
  * **판단:** 유가(CL=F)가 높더라도 정제 마진이 무너지면 정유사(XLE)의 실적이 악화됩니다. 원유 수요 감소의 전조 증상입니다.
- ☑️ **상업용 원유 재고(EIA)의 지속적인 누적**
  * **판단:** 공급이 수요를 초과하기 시작했다는 증거이며, 지표 발표 시마다 유가 하방 압력이 강해집니다.

🟥 공급 과잉 및 생산 단가 하락
- ☑️ **Baker Hughes Rig Count 급증 + 생산자 물가지수(PPI: Drilling) 안정화**
  * **판단:** 시추 비용이 낮아지면서 공급(Rig)이 늘어나는 국면입니다. 셰일 가스/오일 기업 중심인 XOP의 급락을 유발할 수 있습니다.
- ☑️ **SPR 원유 재고의 대규모 방출 지속**
  * **판단:** 정부가 시장 공급을 강제로 늘리는 구간이므로 상방이 제한됩니다.

🟥 역풍(Headwind) 및 자산 가격 붕괴
- ☑️ **달러 인덱스(DXY) 강한 반등 또는 고점 돌파**
  * **판단:** 글로벌 유동성 위축 및 원유 가격의 하락 압력으로 작용합니다.
- ☑️ **CL=F, BZ=F 선물 가격의 데드크로스 (기술적 추세 이탈)**
  * **판단:** 매크로 지표 악화와 함께 선물 가격 가이드라인(예: 50일선, 200일선 이탈)이 깨지면 레버리지 포지션은 즉시 청산해야 합니다.

🟥 인버스 진입 필터
- ☑️ **정제시설 가동률(Utilization) 하락 속 상업용 원유 재고 증가**
  * **판단:** 정유사들이 가동률을 낮추는데도 원유 재고가 쌓인다면 원유 자체의 수요 둔화입니다. 유가 폭락의 신호탄이므로 즉시 현금화하거나 인버스를 준비해야 합니다.
- ☑️ **CL=F, BZ=F 가격 상승 속 Crack Spread 나홀로 급락 (다이버전스)**
  * **판단:** 유가는 오르는데 정제 마진이 깨진다면 제품(휘발유 등) 소비자가 고유가를 버티지 못한다는 뜻입니다. 가짜 상승(Fake)일 확률이 높으며 정유주(XLE)의 급락이 임박했음을 시사합니다.
- ☑️ **Baker Hughes Rig Count 증가 전환 + 원유 선물 콘탱고(Contango) 진입**
  * **판단:** 시추기 수(공급 예고)가 늘어나는데 선물 시장이 콘탱고(원월물>근월물, 공급 과잉 징후)로 바뀐다면, 공급 과잉 우려가 시장을 지배하기 시작했다는 확실한 숏(Short) 신호입니다.
- ☑️ **DXY 급등과 동시에 Drilling PPI 하락 전환**
  * **판단:** 마진의 한계선이었던 생산 비용(PPI)이 내려앉으면서 달러마저 강세로 돌아서는 국면입니다. 에너지 섹터 마진 스프레드가 위아래로 압박받아 주가 폭락을 유발합니다.

---

💡 레버리지 / 인버스 특화 속성 가이드 (XLE vs XOP)

* **XLE**: **`Crack Spread`**와 **`DXY`**의 영향력이 절대적입니다. 정제 마진이 유지되는 한 유가 변동성이 커도 배당과 안정성으로 버티는 경향이 있습니다. 특히 `DXY 급등` + `Crack Spread 낙폭 과대` 시에는 대형주 실적 훼손이 빠르므로 **XLE 인버스**가 안전합니다.
* **XOP**: **`CL=F(WTI)`**, **`Baker Hughes Rig Count`**, **`Drilling PPI`**에 매우 민감합니다. 공급 통제 속에서 유가가 급등하거나 `백워데이션 심화` + `Crack Spread 상승`이 동반될 때 **XOP 레버리지**의 효율이 극대화됩니다. 반대로 공급 과잉 신호(`가동률 하락` + `상업용 재고 증가`) 시에는 **XOP 인버스**의 변동성이 XLE보다 훨씬 큽니다.


### LONG & LEVERAGE

#### ☑️ 단기 수요 및 정제 마진
⬜ **Crack Spread 상승 + 정제시설 가동률(Utilization Rate) 상승**
  * **판단:** 최종 제품(휘발유, 등유 등) 수요가 강력하여 정제 마진이 개선되는 국면입니다. XLE와 XOP 모두에게 강력한 매수 신호입니다.

ℹ️ 설비 가동률 지표의 경우 85% 이하를 유지보수/생산 감축으로 간주하며 90%이상을 공장 풀가동 중으로 간주

In [ ]:
crack_spread = lw.fetch.crack_spread(year=10)
# crack_spread

eia = lw.fetch.eia()
# eia

cs_ur = lw.DualRelation(crack_spread, eia['utilization_rate'])
fig = cs_ur.plotly()
fig.update_layout(height=600)

⬜ **BZ=F(브렌트)와 CL=F(WTI)의 스프레드(Spread) 확대**
  * **판단:** 글로벌 기준물(브렌트)이 미국 유가보다 강하다는 것은 미국 외 전 세계적인 실물 수요가 탄탄하다는 방증입니다. 수출 마진 확대로 미국 에너지 기업에 호재입니다.

In [ ]:
bz_cl = yf.Tickers(['BZ=F', 'CL=F']).history(period='10y', interval='1d')['Close']
bz_cl_spread = bz_cl['BZ=F'] - bz_cl['CL=F']
cs_wb = lw.DualRelation(crack_spread, bz_cl_spread, asset_name='Brent-WTI spread')
fig = cs_wb.plotly()
fig.update_layout(height=600)

⬜  **Crack Spread 상승 속 원유 선물(CL=F, BZ=F) 백워데이션(Backwardation) 심화**
  * **판단:** 근월물 가격이 원월물보다 비싼 현상이 심해진다면, 당장 정제시설에 투입할 실물 원유가 극도로 부족하다는 뜻입니다. 숏 스퀴즈 유발 가능성이 커져 레버리지 진입 최적기입니다.

In [36]:
def get_futures_ticker(current_date, months_ahead):
    # 월과 연도를 안전하게 계산
    total_months = current_date.month - 1 + months_ahead
    target_month = (total_months % 12) + 1
    target_year = current_date.year + (total_months // 12)
    
    kw = {1:'F', 2:'G', 3:'H', 4:'J', 5:'K', 6:'M', 7:'N', 8:'Q', 9:'U', 10:'V', 11:'X', 12:'Z'}
    yy = str(target_year)[-2:]
    
    return f'CL{kw[target_month]}{yy}.NYM'

In [43]:
kw = {1:'F', 2:'G', 3:'H', 4:'J', 5:'K', 6:'M', 7:'N', 8:'Q', 9:'U', 10:'V', 11:'X', 12:'Z'}
yy = str(clock.year)[-2:]

bk1 = get_futures_ticker(clock, 1)
bk3 = get_futures_ticker(clock, 3)
data = yf.Tickers(['CL=F', bk1, bk3]).history(period='10y', interval='1d')['Close']
bk1_spread = data['CL=F'] - data[bk1]
bk3_spread = data['CL=F'] - data[bk3]
rel = lw.DualRelation(bk1_spread, bk3_spread, indicator_name=f'1M Spread({bk1})', asset_name=f'3M Spread({bk3})')
fig = rel.plotly()
fig.add_trace(go.Scatter(
    x=asset['XLE']['close'].index,
    y=asset['XLE']['close'],
    name='XLE',
    xhoverformat='%Y-%m-%d'
))
fig.update_layout(height=600)

[*********************100%***********************]  3 of 3 completed


⬜ **상업용 원유 재고 감소 (EIA)**
  * **판단:** 5년 평균 대비 재고가 하락 추세이거나, 시장 예상치보다 감소 폭이 클 때 단기 유가(CL=F, BZ=F) 상승 압력으로 작용합니다.<br>

⬜ **SPR(전략비축유) 재고 고정 또는 매입 전환 (EIA)**
  * **판단:** 미 정부의 SPR 방출이 끝나고 재충전(매입) 단계로 접어들면 하방 지지선이 형성됩니다.<br>

⬜ **상업용 원유 재고 감소 + SPR 재고 증가 동시 발생 (EIA)**
  * **판단:** 정부(SPR)가 시장에서 원유를 사들이며 민간 재고(상업용)까지 마르는 최상의 공급 부족 신호입니다. 가격 상승의 지속성이 담보됩니다.

In [ ]:
# eia['commercial_stocks']

oil_stocks = lw.DualRelation(eia['commercial_stocks'], eia['SPR_stocks'])
avg_stocks = eia['commercial_stocks'].rolling(window=52*3).mean() # unit: week
fig = oil_stocks.plotly()
fig.add_trace(go.Scatter(
    x=avg_stocks.index,
    y=avg_stocks.values,
    mode='lines',
    name='Avg. Stocks'
))
fig.update_layout(height=600)

#### ☑️ 중장기 공급 및 생산 기조
⬜ **Baker Hughes Rig Count 정체 또는 감소 + 생산자 물가지수(PPI: Drilling) 상승**
  * **판단:** 시추 비용(PPI)은 상승하는데 시추기 수(Rig Count)가 늘지 않는다면, 공급 과잉 우려가 해소되고 고유가가 유지될 환경이 조성됩니다. (특히 XOP에 호재)

In [ ]:
rig_n = lw.fetch.baker_hughes_rig_count() # fetch time: ~20s
today = datetime.now()
start = datetime(today.year - 10, today.month, today.day)
ppi = web.DataReader(['PCU213111213111'], 'fred', start, today)

In [ ]:
# rig_n
# ppi['PCU213111213111']
rig_ppi = lw.DualRelation(rig_n['Total'], ppi['PCU213111213111'], indicator_name='Rig Count', asset_name='PPI')
fig = rig_ppi.plotly()
fig.update_layout(height=600)

#### ☑️ 매크로 및 금융 자산 동향
⬜ **달러 인덱스(DXY) 하락 추세 전환**
  * **판단:** 원유는 달러화로 결제되므로 DXY 하락은 CL=F, BZ=F 가격의 명목적 상승을 유발합니다.

In [ ]:
data = yf.Tickers(['CL=F', 'BZ=F', 'DX-Y.NYB']).history(period='10y', interval='1d')['Close']

dxy = lw.DualRelation(data['DX-Y.NYB'], data['CL=F'], indicator_name='DXY', asset_name='WTI')
fig = dxy.plotly()
fig.add_trace(go.Scatter(
    x=data['BZ=F'].index,
    y=data['BZ=F'].values,
    mode='lines',
    name='Brent',
), secondary_y=True)
fig.update_layout(height=600)

⬜ **천연가스 선물(NG=F)의 계절적 반등 또는 CL=F와의 동반 상승**
  * **판단:** 에너지 섹터 전반의 투자 심리(Sentiment)를 개선시킵니다.

In [ ]:
data = yf.Tickers(['CL=F', 'NG=F']).history(period='10y', interval='1d')['Close']

dxy = lw.DualRelation(data['CL=F'], data['NG=F'], indicator_name='WTI', asset_name='NG')
fig = dxy.plotly()
fig.update_layout(height=600)

In [ ]:
yf.Ticker('CLX26.NYM').history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]

## Price To Earnings

In [ ]:
for ticker, obj in TICKERS.items():
    print(f'{ticker}: {obj.info.get("longName")}')
    print(f'  - TRAILING PE: {obj.info.get("trailingPE")}')
    print(f'  - FORWARD PE: {obj.info.get("forwardPE")}')

## ASSET-DXY

Indicator: DX-Y.NYB: US Dollar Index


<b>[ 핵심 요약 (Core Dynamics) ]</b><br>
일반적으로 <b>역상관 관계(Inverse Correlation)</b> 형성
*   **DXY ↑ (달러 강세)** → 유가 부담 증가 & 위험자산 회피 → **XLE/XOP ↓**
*   **DXY ↓ (달러 약세)** → 원자재 구매력 상승 & 인플레이션 헤지 → **XLE/XOP ↑**

---

<b>[ 자산별 특징 ]</b>

| 자산 | 성격 | DXY와의 민감도 |
| :--- | :--- | :--- |
| **DXY** | 주요 6개국 통화 대비 달러 가치 | 기준점 |
| **XLE** | 대형 에너지주 (엑슨모빌, 쉐브론 등) | 중 (배당·펀더멘털 영향) |
| **XOP** | 중소형 탐사/생산(E&P) 기업 | **고 (유가 변동에 직결)** |

---

<b>[ 일반 vs 예외 시나리오 ]</b>

✅ 일반적 상황: 역상관 관계(Inverse Correlation)
*   **메커니즘**: 달러로 결제되는 원유 특성상, 달러가 비싸지면 수요가 줄어 에너지가 하락함.
*   **배경**: 정상적인 경제 성장기 또는 통화 정책 변화 시.

⚠️ 예외적 상황: 동행 관계 (Positive Correlation)
과거 사례를 통해 본 **DXY와 XLE가 함께 오르는** 특수 환경:

1)  **지정학적 공급 쇼크 (ex. 2022년 우크라이나 전쟁)**
    *   에너지 공급 부족 → 유가 급등 (**XLE ↑**)
    *   안전 자산 선호 & 금리 인상 → 달러 강세 (**DXY ↑**)
2)  **미국 경제의 독주 (Strong US Economy)**
    *   미국 경기 회복 → 에너지 수요 증가 (**XLE ↑**)
    *   타국 대비 높은 성장률/금리 → 달러 유입 (**DXY ↑**)
3)  **초인플레이션 국면**
    *   물가 상승 압력 → 실물 자산 선호 (**XLE ↑**)
    *   인플레이션 방어를 위한 공격적 금리 인상 (**DXY ↑**)

---

<b>[ 투자 시사점 (Checklist) ]</b>
*   **정상기**: 달러의 방향성을 에너지 투자의 선행 지표로 활용.
*   **위기기**: 유가 상승이 '수요' 때문인지 '공급/지정학' 때문인지 구분 (공급 이슈일 땐 달러와 동반 상승 가능).
*   **XOP 주의**: 레버리지 효과가 크므로 달러 강세 국면에서 하락 변동성에 더 취약함.


In [ ]:
dxy = yf.Ticker('DX-Y.NYB').history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
dxy.columns = ['open', 'high', 'low', 'close', 'volume']
dxy = dxy.tz_localize(None) if dxy.index.tz is not None else dxy

### Correlation

In [ ]:
import pylabwons as lw

rel = lw.DualRelation(dxy['close'], asset['XLE']['close'], indicator_name='DXY', asset_name='XLE')
rel.corr

### Plot

In [ ]:
import plotly.io as pio
pio.renderers.default = "vscode"

fig = rel.plotly()
fig.update_layout(height=600)

## ASSET-FUTURES

| 구분 | WTI (서부 텍사스산) | Brent (브렌트유) | Natural Gas |
| :--- | :--- | :--- | :--- |
| **TICKER** | CL=F | BZ=F | NG=F |
| **기준 지역** | 미국 내륙 (텍사스 중심) | 유럽 북해 (해상) | 미국 내륙 |
| **운송 방식** | 파이프라인 (내륙 운송) | 유조선 (해상 운송 유리) | 액화(LNG) 시설 필요,  |
| **주요 시장** | 미국 내 유가 지표 | 글로벌 | 미국 내 가스 지표 |
| **가격 결정** | 글로벌 경기, OPEC+ 감산, 지정학적 리스크 | 좌동 | 미국 내 **기후(한파/폭염)**, 재고 및 LNG 수출률 |
| **변동성** | 매크로 추세 종속 | 좌동 | 기후 변화 등 요인의 **발작적 변동성** |
| **품질** | **최상** (가장 가벼운 경질유) | **우수** (고품질 경질유) | - |

---

<b>[ 핵심 요약 (Core Dynamics) ]</b>
*   <b>원유 선물(CL=F, BZ=F)</b>은 에너지 ETF(**XLE, XOP**)의 방향성을 결정하는 **최선행 지표**입니다.
*   <b>천연가스 선물(NG=F)</b>은 영향력이 상대적으로 낮지만, 가스 비중이 높은 기업의 실적을 가르는 **핵심 변수**입니다.

---

<b>[ 선물 가격과 ETF의 매커니즘 (동행성) ]</b>

| 선물 티커 (Commodity) | 영향받는 ETF (Equity) | 연동 메커니즘 |
| :--- | :--- | :--- |
| **CL=F** (WTI)<br>**BZ=F** (브렌트유) | **XLE** (대형주)<br>**XOP** (중소형주) | • 유가 상승 → 정유사 마진 및 시추 기업 매출 직결 → ETF 상승<br>• **XOP**가 **XLE**보다 원유 선물 가격에 훨씬 더 민감(동행) |
| **NG=F** (천연가스) | **XLE, XOP 내 가스 비중주** | • E&P(탐사/생산) 기업 중 가스 채굴 비중이 높은 기업과 동행<br>• 석유 vs 가스 가격 디커플링 시, 가격 상방 제한 또는 하방 지지 |


---

<b>[ 일반적 관계 vs 예외적 리스크 (디커플링 시나리오) ]</b>

✅ 일반적 상황: 90% 이상의 강력한 동행
*   **현상**: `CL=F/BZ=F 급등 → XOP 폭등 → XLE 상승`
*   선물 시장의 가격 상승이 에너지 기업들의 미래 현금 흐름 기대를 높여 주가를 그대로 끌어올림.

⚠️ 예외적 상황: 선물과 ETF가 따로 노는 경우 (Decoupling)

1.  **석유와 가스의 극단적 디커플링 (가스 폭락)**
    *   **상황**: 유가는 폭등하는데 가스는 미국 내 겨울철 이상 고온으로 재고가 급증해 폭락할 때.
    *   **결과**: 가스 생산 비중이 높은 중소형 E&P 기업들이 타격을 입어, **CL=F가 오름에도 XOP ETF의 상승폭이 제한**됨.
2.  **증시 전체의 유동성 발작 (ex. 팬데믹 초기, 금융위기)**
    *   **상황**: 유가 선물은 지정학적 이유로 버티는데, 주식 시장 전체가 투매 국면일 때.
    *   **결과**: XLE, XOP는 결국 '주식(Equity)' 자산이므로, **선물 가격 지지 여부와 상관없이 지수와 함께 폭락**함.
3.  **정유사 마진(Crack Spread) 악화 및 규제**
    *   **상황**: 국제 유가는 높은 수준을 유지하지만, 경기 침체로 석유 제품 수요가 둔화되거나 정부가 횡재세를 부과할 때.
    *   **결과**: 원유 선물은 상방을 유지해도 대형 정유주 중심의 **XLE ETF는 선행하여 하락**함.

---

<b>[ 투자 시사점 (Checklist) ]</b>
*   **장중 거래**: XLE/XOP 거래 시 실시간으로 움직이는 **CL=F(WTI)의 분봉 추세**를 반드시 싱크하여 모니터링할 것.
*   **XOP 투자 시 필수**: 중소형주 중심인 XOP는 가스 생산 기업이 많으므로, CL=F뿐만 아니라 **NG=F의 추이 및 미국 기후 데이터**를 결합하여 분석해야 안전함.

---

* 현물 가격은 중복/노이즈로 미사용

In [ ]:
cl_f = yf.Ticker('CL=F')
bz_f = yf.Ticker('BZ=F')
ng_f = yf.Ticker('NG=F')
objs = {}
for ticker in [cl_f, bz_f, ng_f]:
    ohlcv = ticker.history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
    ohlcv.columns = ['open', 'high', 'low', 'close', 'volume']
    objs[ticker.ticker] = ohlcv
futures = pd.concat(objs, axis=1)
futures = futures.tz_localize(None) if futures.index.tz is not None else futures

### Correlation

In [ ]:
sym = 'CL=F'
rel = lw.DualRelation(futures[sym]['close'], asset['XOP']['close'], indicator_name=sym, asset_name='XOP')
rel.corr

### Plot

In [ ]:
fig = rel.plotly()
fig.update_layout(height=600)

## ASSET-PPI

Indicator: PCU213111213111: Producer Price Index by Industry: Drilling Oil and Gas Wells @FRED

<br>[ 핵심 요약 (Core Dynamics) ]</b>
*   원유·가스 시추 서비스 및 굴착 장비 운영 상 **생산자 비용(PPI)**
*   투자 유효성: **매우 높음** (중장기 펀더멘털 분석 한정)
*   메커니즘: 유가 상승기 후행하여 시추 비용이 급등 시, <br>에너지 기업의 '마진 압박(Negative)' 혹은 '에너지 사이클의 정점(Peak)' 암시 선행 지표로 활용

---

<br>[ XLE vs XOP에 미치는 비대칭적 영향 ]</br>

이 지수는 기업 규모와 사업 구조에 따라 두 ETF에 완전히 다른 방식으로 영향을 미칩니다.


| 대상 ETF | 지표 상승 시 영향 | 투자 판단 시사점 |
| :--- | :--- | :--- |
| **XLE**<br>(대형 종합 정유사) | **상대적으로 양호 (방어력 높음)** | 자본력 바탕, 시추 비용 상승 방어, 정제 마진(Crack Spread)으로 비용 상쇄 |
| **XOP**<br>(중소형 시추/생산) | **치명적 (수익성 악화 직결)** | 중소형 탐사·생산(E&P) 기업 多, 시추 비용 상승 시 순이익률이 즉각 훼손 |

---

<br>[ 실전 투자 활용 방안 (매매 시나리오) ]</br>

① 사이클 정점 포착: 지수 폭등기 (XOP 비중 축소 신호)
*   **현상**: 유가 선물(`CL=F`)이 높은 수준을 유지하는 상태에서 'Drilling Wells' 지수가 전년 대비 급등할 때.
*   **해석**: 고유가에 취한 기업들이 무리하게 시추 경쟁을 벌여 인력과 장비 비용이 한계치에 도달했음을 의미함 (비용 과부하 단계).
*   **전략**: 에너지 상승 사이클의 후반부일 가능성이 높으므로, 유가 강세에도 불구하고 **XOP(중소형주) 비중을 점진적으로 축소하고 고배당 중심의 XLE로 이동**해야 함.

② 마진 개선 구간 포착: 지수 하락 안정기 (XOP 매수 신호)
*   **현상**: 유가가 박스권이거나 완만하게 회복되는데, 'Drilling Wells' 지수가 낮게 유지되거나 꺾일 때.
*   **해석**: 시추 비용이 저렴해져 1배럴을 캘 때 드는 손익분기점(BEP)이 낮아진 상태입니다. 즉, 유전 기업들의 마진이 극대화되는 구간입니다.
*   **전략**: 중소형 E&P 기업의 실적 서프라이즈 가능성이 커지므로, **XLE보다 유가 레버리지가 큰 XOP ETF를 공격적으로 매수**할 타이밍입니다.

③ 유가(`CL=F`)와 시추 비용 지수의 괴리 추적 (시차 분석)
*   **시차 효과**: 역사적으로 유가 선물 변동은 'Drilling Wells' 지수에 **약 3~6개월 선행**하여 반영됩니다.
*   **활용**: 현재 유가가 폭락했다면 약 3개월 뒤 FRED 지수도 하락할 것을 예측할 수 있으며, 지수가 바닥을 다지는 시점이 에너지 주식(`XOP`, `XLE`)의 장기 가치 투자 진입 시점이 됩니다.

---

<br>[ 최종 행동 가이드 (Checklist) ]</br>
*   **데일리 트레이딩**: 선물 가격(`CL=F`, `BZ=F`)만 보시고 FRED 지수는 완전히 노이즈이므로 무시하십시오.
*   **분기별 포트폴리오 리밸런싱 (추천)**: 매월 중순 FRED 데이터 업데이트 시 전년 동기 대비(YoY) 증가율을 확인하십시오. 비용 증가율이 유가 상승률을 앞지르기 시작하면 **XOP 매도 ➔ XLE나 현금 확보** 전략이 펀더멘털 관점에서 매우 유효합니다.


In [ ]:
today = datetime.now()
start = datetime(today.year - 10, today.month, today.day)
others = web.DataReader(['PCU213111213111'], 'fred', start, today)

### Correlation

In [ ]:
rel = lw.DualRelation(others['PCU213111213111'].dropna(), asset['XLE']['close'], asset_name='XLE', window=24, max_lag=6)
rel.corr

### Plot

In [ ]:
fig = rel.plotly()
fig.update_layout(height=600)

## ASSET-TREASURY

In [ ]:
today = datetime.now()
start = datetime(today.year - 10, today.month, today.day)
others = web.DataReader(['DGS10', 'T10Y2Y', 'BAMLH0A0HYM2'], 'fred', start, today)

### Correlation

In [ ]:
rel = lw.DualRelation(others['BAMLH0A0HYM2'].dropna(), asset['XLE']['close'], asset_name='XLE')
rel.corr

### plot

In [ ]:
fig = rel.plotly()
fig.update_layout(height=600)

## ASSET-RIG COUNT

<b>[ 핵심 요약 (Core Dynamics) ]</b>
*   **지표의 본질**: 현재 미국 내에서 실제로 가동 중인 원유 및 천연가스 시추 장비의 총개수.
*   **투자 유효성**: '매우 높음 (공급 예측 및 업황 판단의 핵심 지표)'
*   **메커니즘**: 유가(`CL=F`) 변동에 약 **3~6개월 후행**하며, Rig Count의 증가는 향후 미국 내 원유 생산량 증가(공급 과잉 리스크)를 암시함.
*   ***데이터 처리**: 원시 데이터 직접 사용 X, YoY / MoM 사용 필수.

Note) 산업의 구조적 변화 반영 (Rig 효율성 급증): 과거 Rig 1기 효율과 현재 Rig 1기 효율의 극명한 차이. 기술 발전(수평 시추, 다정점 시추 등)으로 인해 원시 데이터 상의 Rig Count는 과거보다 훨씬 줄었음에도 미국 원유 생산량은 역대 최고치를 기록하는 현상 발생. 따라서 절대 수치 비교는 무의미, "최근 기조 대비 릭이 늘어나고 있는가 줄어들고 있는가"를 보여주는 YoY 변화율이 업계의 활성도를 정확히 반영

---

<b>[ Rig Count 변화에 따른 XLE vs XOP 역학 관계 ]</b>


| 구분 | XLE (대형 종합 정유사) | XOP (중소형 시추/생산 E&P) |
| :--- | :--- | :--- |
| **Rig Count 상승 시**<br>(시추 활성화) | • 원유 생산 증가로 유가 상방 압박 수혜<br>• 정제 마진 하락 위험 상존 하나 안정적 | • 기업들의 자본지출(CAPEX) 증가 증명<br>• **단기**: 업황 호조 주가 상승<br>• **장기**: 공급 과잉으로 유가 폭락 시 치명타 |
| **Rig Count 하락 시**<br>(시추 위축) | • 글로벌 공급 부족 시 유가 상승 수혜<br>• 장기 유가 지지로 대형주 안정성 부각 | • 자본지출 축소 및 가동 중단 의미<br>• **단기**: 실적 둔화 우려로 주가 하락<br>• **장기**: 공급 부족 유가 급등 시 마진 극대화 |

---

<b>[ 실전 투자 활용 방안 (매매 시나리오) ]</b>

① '자본 효율성' 구간 포착: 유가 상승 + Rig Count 정체 (Best 시나리오)
*   **현상**: 국제 유가(`CL=F`)는 계속 오르는데, 매주 발표되는 Rig Count는 늘어나지 않고 횡보할 때.
*   **해석**: 에너지 기업들이 무리한 증산(CAPEX 지출) 대신, 번 돈으로 **배당 확대 및 자사주 매입**을 하고 있음을 의미함 (미국 셰일 기업들의 주주환원 정책).
*   **전략**: 에너지 섹터의 황금기입니다. **XLE(고배당)와 XOP(수익성 극대화) 모두 강력 매수 및 홀딩** 전략이 유효합니다.

② '치킨 게임/치명적 과열' 구간 포착: 유가 정체 + Rig Count 폭등 (Exit 신호)
*   **현상**: 유가는 박스권에 갇혀 있거나 완만하게 하락하는데, Rig Count가 전고점을 뚫고 계속 늘어날 때.
*   **해석**: 무리한 증산 경쟁으로 몇 달 뒤 시장에 원유가 쏟아져 나와 유가 폭락을 유발할 수 있는 시그널입니다.
*   **전략**: **XOP(중소형주)를 즉각 매도**하거나 비중을 최소화해야 합니다. 공급 과잉 충격은 부채 비율이 높은 중소형 E&P 기업에 가장 먼저 타격을 줍니다.

③ 유가(`CL=F`)와 Rig Count의 시차(Time Lag)를 활용한 바닥 잡기
*   **현상**: 유가가 바닥을 치고 반등하기 시작해도, Rig Count는 현장 준비 기간 때문에 수개월간 계속 감소함.
*   **해석**: Rig Count가 계속 줄어든다는 것은 향후 수개월간 공급이 계속 정체될 것임을 의미하므로 유가 반등의 신뢰도가 높아짐.
*   **전략**: 유가가 반등 흐름인데 Rig Count가 여전히 바닥에서 기고 있다면, 이는 **XOP와 XLE의 장기 가치 투자 진입을 위한 최적의 적기**입니다.

---

<b>[ 최종 행동 가이드 (Checklist) ]</b>
*   **데이터 확인 주기**: 매주 금요일 오후 1시(미국 동부 시간) Baker Hughes사 웹사이트 또는 FRED(`BRSRCNT`, `BARRCNT`)를 통해 전주 대비 증감세를 체크하십시오.
*   **동시 지표 결합**: 앞서 정리한 **Drilling Wells 비용 지수(PPI)**와 결합하십시오. Rig Count는 늘어나는데 시추 비용(PPI)까지 급등하고 있다면 에너지 기업들의 마진이 깎이는 국면이므로 주가 상방이 제한됩니다.


### Correlation

In [ ]:
rig_count = lw.fetch.baker_hughes_rig_count()
rig_count = lw.tools.to_mom_yoy(rig_count['Total'], fill_method='ffill')

rel = lw.DualRelation(rig_count['Total'].dropna(), asset['XLE']['close'], indicator_name='RigCount', asset_name='XLE')
rel.corr()

In [ ]:
import plotly.graph_objects as go
fig = rel.plotly()
fig.update_layout(height=600)

# yoy = test['']
fig.add_trace(
    go.Scatter(x=test.index, y=test['YoY'], mode='lines', name='RigCount YoY(%)', showlegend=True, visible='legendonly', 
               xhoverformat='%Y-%m-%d',
               hovertemplate='YoY: %{y:.2f}%<extra></extra>')
)